In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-5 : GET RAW INVALID RECORDS FROM QUARANTINE
# =========================================================

invalid_df = spark.read.table("retails.silver.departments_quarantine") \
                    .filter((col("_rescued_data").isNotNull()) & (col("quarantine_status") == 'NEW'))

In [0]:
from pyspark.sql.functions import col, get_json_object as get_json_from

# =========================================================
# STEP-6 : TRY TO RECOVER RESCUED COLUMNS
# =========================================================
recovered_df = invalid_df \
                .withColumn("department_id_fixed", get_json_from(col("_rescued_data"), "$.department_id")) \
                .withColumn("department_name_fixed", get_json_from(col("_rescued_data"), "$.department_name"))


In [0]:
from pyspark.sql.functions import coalesce, trim

# =========================================================
# STEP-7 : MERGE RECOVERED VALUES
# =========================================================

recovered_df = recovered_df \
    .withColumn(
        "department_id",
        coalesce(col("department_id"), col("department_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "department_name",
        coalesce(col("department_name"), trim(col("department_name_fixed")))
    ) 
    

In [0]:
recovered_df = recovered_df.drop("department_id_fixed", "department_name_fixed")

In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-8 : APPLY DATA QUALITY RULES
# =========================================================

cleaned_recovered_df = recovered_df.filter(
    col("department_id").isNotNull() &
    col("department_name").isNotNull() 
)

In [0]:
cleaned_recovered_df = cleaned_recovered_df.dropDuplicates(["department_id"])
cleaned_recovered_df.createOrReplaceTempView("departments_cleaned_vw_fixed")

In [0]:
from pyspark.sql.functions import when, current_timestamp, sha2, concat_ws

cleaned_recovered_df = cleaned_recovered_df \
    .withColumn("department_id", col("department_id").cast("bigint")) \
    .withColumn("batch_id", col("batch_id").cast("integer")) \
    .withColumn("is_deleted", when(col("op")=='DELETE', True).otherwise(False)) 


cleaned_recovered_df = cleaned_recovered_df.select("department_id", "department_name", "op", "is_deleted", "source_system", "source_file_name", "ingestion_ts", "ingestion_dt", "batch_id", "run_id")

cleaned_recovered_df = cleaned_recovered_df.withColumn("event_ts", current_timestamp()) \
                    .withColumn("record_hash",
                                sha2(
                                    concat_ws(
                                        "||",
                                        col("department_id"),
                                        col("department_name")
                                    ),
                                    256
                                )
                            )
    

try:
    cleaned_recovered_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable("retails.silver.departments_cdc")

except Exception as e:
    print(str(e))

In [0]:
merge_query_rescued = """
    MERGE INTO retails.silver.departments_quarantine t
    USING departments_cleaned_vw_fixed s
    ON t.department_id = s.department_id
    WHEN MATCHED THEN
        UPDATE SET t.quarantine_status = 'FIXED', t.reprocessed_at = current_timestamp()
 
    """
spark.sql(merge_query_rescued).show()


In [0]:
update_query_corrupt = """
        UPDATE retails.silver.departments_quarantine
        SET
            quarantine_status = 'INVALID',
            reprocessed_at = current_timestamp()
        WHERE quarantine_status = 'NEW'
"""

spark.sql(update_query_corrupt).show()

In [0]:
%sql
-- select * from retails.silver.departments_quarantine;
-- select * from retails.silver.departments_cdc;
